<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026psy3a_lect04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 心理学特講IIIA 第4回：Flanker課題

**担当**: 浅川伸一  
**2026年度 前期**

---

## 本日の実習内容

1. Flanker課題の設計
2. 模擬データの生成
3. 記述統計と可視化
4. 一要因分散分析（ANOVA）
5. 多重比較（Tukey法）と効果量
6. **3つの干渉課題の統合比較**（Stroop・Simon・Flanker）
7. 注意の空間的範囲の分析
8. 準備学習課題

---

### 第4回までの積み上げ

| 回 | 課題 | 統計手法 | 干渉の性質 |
|----|------|---------|----------|
| 第2回 | Stroop | 一要因ANOVA | 意味的・言語的 |
| 第3回 | Simon | 対応のあるt検定 | 空間的 |
| **第4回** | **Flanker** | **一要因ANOVA** | **知覚的（周辺刺激）** |

本日はFlanker課題の分析に加え，**3課題を統合して比較**します。

---
## 0. 環境準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, ttest_rel
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import AnovaRM

!pip install japanize-matplotlib --quiet
import japanize_matplotlib

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("環境準備完了")

---
## 1. Flanker課題の設計

### 1.1 課題の構造

**刺激配置**
```
← ← ← ← ←   一致条件（Congruent）：中央と周辺が同じ方向
← ← → ← ←   不一致条件（Incongruent）：中央と周辺が逆方向
. . → . .    中立条件（Neutral）：周辺は中立記号
```

**課題**：中央の矢印の方向（←/→）に応じてキーを押す

**Flanker効果** = RT(不一致) − RT(一致)

In [ ]:
# 刺激の定義
stimuli = {
    'congruent':   {'display': '→ → → → →',  'label': '一致'},
    'neutral':     {'display': '. . → . .',   'label': '中立'},
    'incongruent': {'display': '← ← → ← ←', 'label': '不一致'},
}

print("=== Flanker課題の刺激 ===")
print(f"{'条件':<12} {'刺激':<22} {'ターゲット'} {'Flanker'}")
print("-" * 55)
for key, val in stimuli.items():
    flanker_desc = ('同じ方向' if key == 'congruent'
                    else '中立' if key == 'neutral'
                    else '逆方向')
    print(f"{val['label']:<10} {val['display']:<22} →        {flanker_desc}")

print("\n課題：中央の矢印（→/←）の方向にキーを押す")
print("Flanker刺激の方向は課題に無関連だが，反応に影響する")

---
## 2. 模擬データの生成

In [ ]:
def generate_flanker_data(n_trials=40, n_participants=1, seed=42):
    """
    Flanker課題の模擬データを生成する

    Parameters
    ----------
    n_trials      : 各条件の試行数
    n_participants: 参加者数
    seed          : 乱数シード

    Returns
    -------
    df : pandas.DataFrame
    """
    rng = np.random.default_rng(seed)
    data = []

    # 条件ごとのパラメータ（文献に基づく典型的な値）
    rt_params = {
        'congruent':   {'mean': 460, 'std': 65},
        'neutral':     {'mean': 510, 'std': 70},
        'incongruent': {'mean': 560, 'std': 75},
    }
    error_rate = {'congruent': 0.02, 'neutral': 0.03, 'incongruent': 0.06}

    conditions = ['congruent', 'neutral', 'incongruent']

    for pid in range(1, n_participants + 1):
        # 参加者ごとの個人差（ランダム切片）
        individual_offset = rng.normal(0, 30)
        trial_counter = 1

        for cond in conditions:
            p = rt_params[cond]
            for _ in range(n_trials):
                rt = rng.normal(p['mean'] + individual_offset, p['std'])
                rt = max(250, rt)  # 最小値250ms
                acc = int(rng.random() > error_rate[cond])
                if acc == 0:
                    rt *= 0.88  # エラー試行は速い傾向
                data.append({
                    'participant': pid,
                    'trial':      trial_counter,
                    'condition':  cond,
                    'rt':         round(rt, 1),
                    'accuracy':   acc
                })
                trial_counter += 1

    df = pd.DataFrame(data)
    df['condition'] = pd.Categorical(
        df['condition'],
        categories=['congruent', 'neutral', 'incongruent'],
        ordered=True
    )
    return df

# データ生成
df_fl = generate_flanker_data(n_trials=40)
df_fl_ok = df_fl[df_fl['accuracy'] == 1].copy()

print(f"生成試行数 : {len(df_fl)}")
print(f"正答試行数 : {len(df_fl_ok)}")
print(f"全体正答率 : {df_fl['accuracy'].mean()*100:.1f}%")
print()
print(df_fl.head(9).to_string(index=False))

---
## 3. 記述統計

In [ ]:
label_map = {'congruent': '一致', 'neutral': '中立', 'incongruent': '不一致'}

desc = df_fl_ok.groupby('condition')['rt'].agg(
    試行数='count', 平均='mean', SD='std',
    SE=lambda x: x.std() / np.sqrt(len(x)),
    中央値='median', 最小値='min', 最大値='max'
).round(2)

print("=== 条件別反応時間（正答試行のみ，ms）===")
print(desc)

# Flanker効果
rt_mean = df_fl_ok.groupby('condition')['rt'].mean()
fe_vs_neutral     = rt_mean['incongruent'] - rt_mean['neutral']
fe_vs_congruent   = rt_mean['incongruent'] - rt_mean['congruent']
facilitation      = rt_mean['neutral']     - rt_mean['congruent']

print("\n=== Flanker効果 ===")
print(f"不一致 − 一致   = {fe_vs_congruent:.1f} ms  （Flanker効果・広義）")
print(f"不一致 − 中立   = {fe_vs_neutral:.1f} ms   （干渉効果）")
print(f"中立   − 一致   = {facilitation:.1f} ms   （促進効果）")

# 条件別正答率
acc = df_fl.groupby('condition')['accuracy'].mean() * 100
print("\n=== 条件別正答率（%）===")
for cond, val in acc.items():
    print(f"  {label_map[cond]}: {val:.1f}%")

---
## 4. データ可視化

In [ ]:
conditions_jp = ['一致', '中立', '不一致']
conditions_en = ['congruent', 'neutral', 'incongruent']
palette = ['#66c2a5', '#fc8d62', '#8da0cb']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- 箱ひげ図 ---
data_list = [df_fl_ok[df_fl_ok['condition'] == c]['rt'].values
             for c in conditions_en]
bp = axes[0].boxplot(data_list, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_xticklabels(conditions_jp)
axes[0].set_ylabel('反応時間（ms）', fontsize=11)
axes[0].set_title('箱ひげ図', fontsize=12)

# --- バイオリンプロット ---
df_fl_ok_plot = df_fl_ok.copy()
df_fl_ok_plot['条件'] = df_fl_ok_plot['condition'].map(label_map)
df_fl_ok_plot['条件'] = pd.Categorical(
    df_fl_ok_plot['条件'], categories=['一致', '中立', '不一致'], ordered=True)
sns.violinplot(data=df_fl_ok_plot, x='条件', y='rt',
               palette=palette, ax=axes[1], inner='box')
axes[1].set_ylabel('反応時間（ms）', fontsize=11)
axes[1].set_title('バイオリンプロット', fontsize=12)

# --- 平均値 ± SE ---
means = desc['平均'].values
ses   = desc['SE'].values
xpos  = np.arange(3)
bars  = axes[2].bar(xpos, means, yerr=ses, capsize=8,
                    alpha=0.8, color=palette)
axes[2].set_xticks(xpos)
axes[2].set_xticklabels(conditions_jp)
axes[2].set_ylabel('平均反応時間（ms）', fontsize=11)
axes[2].set_title('平均値 ± 標準誤差', fontsize=12)
axes[2].set_ylim(0, max(means) * 1.35)

# Flanker効果の両矢印
y_ann = max(means) * 1.25
axes[2].annotate('', xy=(2, y_ann), xytext=(0, y_ann),
                 arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
axes[2].text(1, y_ann * 1.02,
             f'Flanker効果\n{fe_vs_congruent:.0f} ms',
             ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Flanker課題：条件別反応時間（正答試行）', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. 一要因分散分析（One-Way ANOVA）

**なぜANOVA？**  
3条件の比較 → t検定を繰り返すと第一種過誤が増加  
→ 一要因ANOVAで全体の差を検定してから多重比較

In [ ]:
rt_con = df_fl_ok[df_fl_ok['condition'] == 'congruent']['rt'].values
rt_neu = df_fl_ok[df_fl_ok['condition'] == 'neutral']['rt'].values
rt_inc = df_fl_ok[df_fl_ok['condition'] == 'incongruent']['rt'].values

F, p = f_oneway(rt_con, rt_neu, rt_inc)

# 効果量 η²
all_rt   = np.concatenate([rt_con, rt_neu, rt_inc])
grand_m  = all_rt.mean()
ss_between = sum(
    len(g) * (g.mean() - grand_m) ** 2
    for g in [rt_con, rt_neu, rt_inc]
)
ss_total = np.sum((all_rt - grand_m) ** 2)
eta2 = ss_between / ss_total

df_between = 2                              # k - 1
df_within  = len(all_rt) - 3               # N - k

print("=== 一要因分散分析（One-Way ANOVA）===")
print(f"F({df_between}, {df_within}) = {F:.4f}")
print(f"p = {p:.6f}")
print(f"η² = {eta2:.4f}")
print()
print("η²の目安（Cohen, 1988）:")
print("  小: η² = 0.01  中: η² = 0.06  大: η² = 0.14")
size = ('大' if eta2 >= 0.14 else '中' if eta2 >= 0.06 else '小')
print(f"\n今回の効果量は「{size}」に相当")

sig = p < 0.05
print(f"\n結論: 有意水準5%で {'有意差あり ✓' if sig else '有意差なし'}")

### 5.2 多重比較：Tukey HSD法

In [ ]:
# Tukey HSD
all_rt_arr   = np.concatenate([rt_con, rt_neu, rt_inc])
all_cond_arr = (['congruent']   * len(rt_con) +
                ['neutral']     * len(rt_neu) +
                ['incongruent'] * len(rt_inc))

tukey = pairwise_tukeyhsd(all_rt_arr, all_cond_arr, alpha=0.05)
print("=== Tukey HSD 多重比較 ===")
print(tukey)

# Cohen's d（各ペア）
def cohens_d(a, b):
    pooled_sd = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    return (a.mean() - b.mean()) / pooled_sd

pairs = [
    ('一致 vs 中立',    rt_con, rt_neu),
    ('中立 vs 不一致',  rt_neu, rt_inc),
    ('一致 vs 不一致',  rt_con, rt_inc),
]
print("\n=== ペアごとの効果量（Cohen's d）===")
for label, a, b in pairs:
    d = cohens_d(b, a)   # 遅い - 速い
    size = ('大' if abs(d) >= 0.8 else '中' if abs(d) >= 0.5 else '小')
    print(f"  {label:<20}: d = {d:.4f}（{size}）")

---
## 6. 3つの干渉課題の統合比較

**第2回（Stroop），第3回（Simon），第4回（Flanker）**のデータを統合し，
3課題の干渉効果の大きさを比較します。

### 6.1 Stroopデータの生成（第2回の再現）

In [ ]:
def generate_stroop_data(n_trials=40, seed=42):
    rng = np.random.default_rng(seed)
    rt_params = {
        'congruent':   {'mean': 600, 'std': 90},
        'neutral':     {'mean': 700, 'std': 95},
        'incongruent': {'mean': 850, 'std': 100},
    }
    data = []
    for cond, p in rt_params.items():
        for _ in range(n_trials):
            rt  = max(300, rng.normal(p['mean'], p['std']))
            acc = int(rng.random() > 0.05)
            data.append({'condition': cond, 'rt': round(rt, 1), 'accuracy': acc})
    return pd.DataFrame(data)

df_stroop = generate_stroop_data(n_trials=40)
df_stroop_ok = df_stroop[df_stroop['accuracy'] == 1]
print("Stroopデータ生成完了:", len(df_stroop_ok), "試行")

In [ ]:
# Simonデータの生成（第3回の再現）
def generate_simon_data(n_trials=40, seed=42):
    rng = np.random.default_rng(seed)
    rt_params = {
        'congruent':   {'mean': 550, 'std': 70},
        'incongruent': {'mean': 650, 'std': 80},
    }
    data = []
    for cond, p in rt_params.items():
        for _ in range(n_trials):
            rt  = max(300, rng.normal(p['mean'], p['std']))
            acc = int(rng.random() > 0.04)
            data.append({'condition': cond, 'rt': round(rt, 1), 'accuracy': acc})
    return pd.DataFrame(data)

df_simon = generate_simon_data(n_trials=40)
df_simon_ok = df_simon[df_simon['accuracy'] == 1]
print("Simonデータ生成完了:", len(df_simon_ok), "試行")

### 6.2 干渉効果の算出

In [ ]:
def calc_interference(df_ok, baseline='congruent', target='incongruent'):
    """干渉効果（不一致 − 一致）を計算"""
    m = df_ok.groupby('condition')['rt'].mean()
    return m[target] - m[baseline]

def calc_facilitation(df_ok, baseline='neutral', fast='congruent'):
    """促進効果（中立 − 一致）を計算"""
    m = df_ok.groupby('condition')['rt'].mean()
    return m[baseline] - m[fast]

# 各課題の指標
stroop_interf = calc_interference(df_stroop_ok)
stroop_facil  = calc_facilitation(df_stroop_ok)
simon_interf  = calc_interference(df_simon_ok)
flanker_interf = calc_interference(df_fl_ok)
flanker_facil  = calc_facilitation(df_fl_ok)

comparison = pd.DataFrame({
    '課題':     ['Stroop', 'Simon', 'Flanker'],
    '干渉の性質': ['意味的・言語的', '空間的', '知覚的（周辺刺激）'],
    '一致RT(ms)': [
        df_stroop_ok[df_stroop_ok['condition']=='congruent']['rt'].mean(),
        df_simon_ok[df_simon_ok['condition']=='congruent']['rt'].mean(),
        rt_mean['congruent']
    ],
    '不一致RT(ms)': [
        df_stroop_ok[df_stroop_ok['condition']=='incongruent']['rt'].mean(),
        df_simon_ok[df_simon_ok['condition']=='incongruent']['rt'].mean(),
        rt_mean['incongruent']
    ],
    '干渉効果(ms)': [stroop_interf, simon_interf, flanker_interf],
})

comparison['促進効果(ms)'] = [
    stroop_facil, np.nan, flanker_facil
]

print("=== 3課題の干渉効果比較 ===")
print(comparison.round(1).to_string(index=False))

### 6.3 統合比較の可視化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tasks     = ['Stroop', 'Simon', 'Flanker']
colors_t  = ['#e41a1c', '#377eb8', '#4daf4a']

# --- 左：一致vs不一致の平均RT ---
con_means = [
    df_stroop_ok[df_stroop_ok['condition']=='congruent']['rt'].mean(),
    df_simon_ok[df_simon_ok['condition']=='congruent']['rt'].mean(),
    rt_mean['congruent']
]
inc_means = [
    df_stroop_ok[df_stroop_ok['condition']=='incongruent']['rt'].mean(),
    df_simon_ok[df_simon_ok['condition']=='incongruent']['rt'].mean(),
    rt_mean['incongruent']
]

x   = np.arange(len(tasks))
w   = 0.35
axes[0].bar(x - w/2, con_means, w, label='一致', alpha=0.8, color='#66c2a5')
axes[0].bar(x + w/2, inc_means, w, label='不一致', alpha=0.8, color='#fc8d62')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tasks)
axes[0].set_ylabel('平均反応時間（ms）', fontsize=11)
axes[0].set_title('一致・不一致条件の比較', fontsize=12)
axes[0].legend()

# --- 中：干渉効果の大きさ ---
interf = [stroop_interf, simon_interf, flanker_interf]
bars   = axes[1].bar(tasks, interf, alpha=0.8, color=colors_t)
for bar, val in zip(bars, interf):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 3, f'{val:.0f} ms',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('干渉効果（ms）\n不一致RT − 一致RT', fontsize=11)
axes[1].set_title('干渉効果の大きさ比較', fontsize=12)

# --- 右：干渉の性質の比較（レーダーチャート風の棒グラフ）---
dims = ['意味的処理\nの関与', '空間的処理\nの関与', '知覚的\n競合']
scores = {
    'Stroop':  [10, 3, 4],
    'Simon':   [1,  10, 3],
    'Flanker': [2,  5,  10],
}
xs = np.arange(len(dims))
wd = 0.25
for i, (task, sc) in enumerate(scores.items()):
    axes[2].bar(xs + i*wd - wd, sc, wd,
                label=task, alpha=0.8, color=colors_t[i])
axes[2].set_xticks(xs)
axes[2].set_xticklabels(dims, fontsize=9)
axes[2].set_ylabel('関与の程度（相対値）', fontsize=11)
axes[2].set_title('干渉の性質の比較（模式図）', fontsize=12)
axes[2].legend()

plt.suptitle('Stroop・Simon・Flanker 3課題の統合比較', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. 注意の空間的範囲の分析

**Eriksen & Eriksen (1974)の主要な発見**：  
ターゲットとFlankerの間隔（SOA）が大きいほど干渉が小さくなる。

この性質をシミュレーションで確認します。

In [ ]:
# 間隔ごとのFlanker効果のシミュレーション
# 間隔が大きいほど干渉が小さくなるモデル
rng = np.random.default_rng(42)

# 距離（degree of visual angle）
distances = [0.06, 0.12, 0.25, 0.50, 1.00]  # Eriksen & Eriksen (1974)の設定

# 距離と干渉効果の関係（指数的減衰モデル）
# FE(d) = A * exp(-λ * d) + noise
A    = 100.0   # 最大干渉効果
lam  = 1.8     # 減衰率

results_spatial = []
for d in distances:
    fe_true = A * np.exp(-lam * d)
    # 複数参加者のばらつきをシミュレート
    fe_obs  = rng.normal(fe_true, 12, 20)  # n=20
    results_spatial.append({
        '距離（度）': d,
        '平均Flanker効果(ms)': fe_obs.mean(),
        'SE':         fe_obs.std() / np.sqrt(len(fe_obs))
    })

df_spatial = pd.DataFrame(results_spatial)
print("=== 距離と干渉効果（シミュレーション）===")
print(df_spatial.round(2).to_string(index=False))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 実データ（距離ごとの干渉効果）
axes[0].errorbar(
    df_spatial['距離（度）'],
    df_spatial['平均Flanker効果(ms)'],
    yerr=df_spatial['SE'] * 1.96,
    marker='o', markersize=8, capsize=5,
    linewidth=2, color='steelblue', label='シミュレーション'
)

# 理論曲線
d_smooth = np.linspace(0.04, 1.1, 200)
axes[0].plot(d_smooth, A * np.exp(-lam * d_smooth),
             'r--', linewidth=1.5, label='指数減衰モデル')
axes[0].set_xlabel('ターゲット−Flanker間距離（視角度）', fontsize=11)
axes[0].set_ylabel('Flanker効果（ms）', fontsize=11)
axes[0].set_title('距離とFlanker干渉効果\n（距離が大きいほど干渉は小）', fontsize=12)
axes[0].legend()
axes[0].set_xlim(0, 1.1)
axes[0].set_ylim(0, None)

# 注意のスポットライトの模式図
theta = np.linspace(0, 2*np.pi, 200)
for r, alpha, label in [(0.3, 0.4, '狭い焦点'), (0.7, 0.2, '広い焦点')]:
    axes[1].fill(r*np.cos(theta), r*np.sin(theta),
                 alpha=alpha, color='yellow', label=label)
axes[1].plot(0, 0, 'r*', markersize=15, label='ターゲット')
for x_f, label_f in [(-0.3, 'F'), (-0.15, 'F'),
                       (0.15, 'F'), (0.3, 'F')]:
    axes[1].plot(x_f, 0, 'bs', markersize=10)
    axes[1].text(x_f, -0.12, 'F', ha='center', fontsize=10, color='blue')
axes[1].set_xlim(-1, 1)
axes[1].set_ylim(-1, 1)
axes[1].set_aspect('equal')
axes[1].set_title('注意のスポットライトモデル\n（広い焦点 → 近いFlankerも処理される）', fontsize=12)
axes[1].legend(loc='upper right', fontsize=9)
axes[1].axis('off')

plt.suptitle('Flanker干渉と注意の空間的範囲', fontsize=13)
plt.tight_layout()
plt.show()

print("\n解釈：")
print("  近いFlanker → 注意の焦点内に入りやすい → 干渉大")
print("  遠いFlanker → 注意の焦点外 → 干渉小")
print("  → 注意は空間的な広がりを持つ（スポットライトモデル）")

---
## 8. 結果のまとめ

In [ ]:
print("=" * 55)
print("  第4回：Flanker課題 まとめ")
print("=" * 55)

print("\n【Flanker課題の結果】")
for c, m, se in zip(conditions_jp, desc['平均'].values, desc['SE'].values):
    print(f"  {c}条件: {m:.1f} ms (SE={se:.1f})")
print(f"  Flanker効果（不一致−一致）: {fe_vs_congruent:.1f} ms")
print(f"  ANOVA: F({df_between},{df_within}) = {F:.2f}, p = {p:.4f}, η² = {eta2:.3f}")

print("\n【3課題の干渉効果の大きさ】")
for task, ie in zip(tasks, [stroop_interf, simon_interf, flanker_interf]):
    print(f"  {task:<8}: {ie:.1f} ms")
print(f"  大小関係: Stroop > Simon > Flanker (本データ)")

print("\n【注意の空間的範囲】")
print("  ターゲット−Flanker間距離↑ → 干渉効果↓")
print("  → 注意のスポットライトモデルで説明可能")

print("\n【3課題の共通点と相違点】")
print("  共通点: 課題無関連情報の自動処理による反応選択の競合")
print("  Stroop: 意味的・言語的処理の自動性")
print("  Simon : 空間的位置の自動コーディング")
print("  Flanker: 注意焦点外の刺激の自動処理")

---
## 準備学習課題

**課題1**（100字程度・必須）  
Flanker効果とは何か，選択的注意の観点から説明せよ。

**課題2**（100字程度・必須）  
Stroop・Simon・Flankerの3課題に共通する認知的メカニズムを説明し，
それぞれの干渉の性質の違いを述べよ。

**課題3**（任意）  
距離とFlanker効果の関係から，注意のスポットライトモデルを支持する根拠を説明せよ。

---

### 解答欄

**課題1**:


**課題2**:


**課題3（任意）**:


---
## 参考文献

1. Eriksen, B. A., & Eriksen, C. W. (1974). Effects of noise letters upon the identification of a target letter in a nonsearch task. *Perception & Psychophysics, 16*(1), 143–149.

2. Eriksen, C. W., & St. James, J. D. (1986). Visual attention within and around the field of focal attention: A zoom lens model. *Perception & Psychophysics, 40*(4), 225–240.

3. Stroop, J. R. (1935). Studies of interference in serial verbal reactions. *Journal of Experimental Psychology, 18*(6), 643–662.

4. Simon, J. R. (1969). Reactions toward the source of stimulation. *Journal of Experimental Psychology, 81*(1), 174–176.

---
*心理学特講IIIA 第4回実習ノートブック | 担当：浅川伸一*